# Oikolab Proxy Extraction

Downloads hourly weather data from the Oikolab ERA5 API for a given location and date range,
cleans the response, and saves the result in the format expected by **Notebook 01**.

The Oikolab API enforces a 500-unit limit per request.
With 10 parameters the limit is ~120 units per 6-month chunk, well within bounds.

---

## Output Format Contract

| Requirement | Detail |
|---|---|
| **File** | `../data/raw/proxies/oikolab_weather.csv` (relative to `auxiliary/`) |
| **Index** | `datetime` column (UTC timestamps), first CSV column |
| **Read by NB01** | `pd.read_csv(..., index_col=0, parse_dates=True)` |
| **Metadata dropped** | `coordinates (lat,lon)`, `model (name)`, `model elevation (surface)`, `utc_offset (hrs)` |
| **Data columns** | 10 SHM-relevant ERA5 weather variables |
| **Timezone** | UTC - align to local time in Notebook 01 if the sensor uses local time |

---

## Parameters downloaded

| Variable | Role in SHM |
|---|---|
| `temperature` | Primary driver of thermal expansion/contraction |
| `relative_humidity` | Governs masonry moisture content and hygric deformation |
| `dewpoint_temperature` | Condensation threshold on surfaces |
| `surface_solar_radiation` | Thermal forcing on sun-exposed facades |
| `surface_thermal_radiation` | Longwave radiative cooling at night |
| `total_precipitation` | Direct moisture input to the wall |
| `wind_speed` | Wind-driven rain and evaporative cooling |
| `wind_direction` | Driving rain orientation |
| `skin_temperature` | Closest ERA5 proxy to surface-mounted sensor readings |
| `snowfall` | Freeze-thaw loading; relevant for Apennine winters |

---

## Steps
1. **Parameters** - Location, date range, API key, output path.
2. **Download** - Fetch one 6-month window at a time, with progress reporting.
3. **Concatenate & clean** - Merge chunks, promote datetime index, drop metadata columns.
4. **Save** - Write CSV with `index=True`.

## Step 1 - Parameters

| Parameter | Description |
|---|---|
| `LAT`, `LON` | Decimal coordinates of the monitoring site |
| `START`, `END` | Inclusive date range (`YYYY-MM-DD`). Should fully bracket the sensor window |
| `API_KEY` | Oikolab API key |
| `OUTPUT_PATH` | Destination CSV - must match `PROXY_FILE` in Notebook 01 |
| `CHUNK_MONTHS` | Request window size in months. Reduce if you hit 500-unit errors |

In [2]:
import requests
import pandas as pd
import io
import os
import time
from datetime import date
from dateutil.relativedelta import relativedelta

# === USER INPUT ===
LAT          = 43.35343
LON          = 12.582047
START        = '2018-01-01'
END          = '2026-04-25'
API_KEY      = '97efb218e1814a47abfff249488f6dfc'
OUTPUT_PATH  = '../data/raw/proxies/oikolab_weather.csv'
CHUNK_MONTHS = 6   # 6 months x 10 params ~ 60 units (limit is 500)
# ==================

META_COLS = [
    'coordinates (lat,lon)',
    'model (name)',
    'model elevation (surface)',
    'utc_offset (hrs)',
]

## Step 2 - Download

10 SHM-relevant regressors. With 6-month chunks each request uses ~60 units.
A 1-second pause between requests avoids rate-limiting.

In [3]:
PARAMS = [
    'temperature',
    'relative_humidity',
    'dewpoint_temperature',
    'surface_solar_radiation',
    'surface_thermal_radiation',
    'total_precipitation',
    'wind_speed',
    'wind_direction',
    'skin_temperature',
    'snowfall',
]

def build_chunk_windows(start_str, end_str, months):
    """Yield (chunk_start, chunk_end) pairs of `months` months each."""
    start = date.fromisoformat(start_str)
    end   = date.fromisoformat(end_str)
    cursor = start
    while cursor <= end:
        chunk_end = min(cursor + relativedelta(months=months) - relativedelta(days=1), end)
        yield cursor.isoformat(), chunk_end.isoformat()
        cursor = cursor + relativedelta(months=months)

windows = list(build_chunk_windows(START, END, CHUNK_MONTHS))
print(f'Fetching {len(windows)} chunks of {CHUNK_MONTHS} months ({len(PARAMS)} params each) ...')

chunks = []
for i, (chunk_start, chunk_end) in enumerate(windows):
    print(f'  [{i+1}/{len(windows)}] {chunk_start} -> {chunk_end} ...', end=' ', flush=True)
    r = requests.get(
        'https://api.oikolab.com/weather',
        params={
            'param': PARAMS,
            'lat': LAT,
            'lon': LON,
            'start': chunk_start,
            'end': chunk_end,
            'freq': 'H',
            'resample_method': 'mean',
            'format': 'csv',
        },
        headers={'api-key': API_KEY},
    )
    if not r.ok:
        print(f'FAILED ({r.status_code})')
        print(r.text)
        r.raise_for_status()
    chunk_df = pd.read_csv(io.StringIO(r.text))
    chunks.append(chunk_df)
    print(f'OK ({len(chunk_df)} rows)')
    time.sleep(1)

print(f'All {len(windows)} chunks downloaded.')

Fetching 17 chunks of 6 months (10 params each) ...
  [1/17] 2018-01-01 -> 2018-06-30 ... OK (4321 rows)
  [2/17] 2018-07-01 -> 2018-12-31 ... OK (4393 rows)
  [3/17] 2019-01-01 -> 2019-06-30 ... OK (4321 rows)
  [4/17] 2019-07-01 -> 2019-12-31 ... OK (4393 rows)
  [5/17] 2020-01-01 -> 2020-06-30 ... OK (4345 rows)
  [6/17] 2020-07-01 -> 2020-12-31 ... OK (4393 rows)
  [7/17] 2021-01-01 -> 2021-06-30 ... OK (4321 rows)
  [8/17] 2021-07-01 -> 2021-12-31 ... OK (4393 rows)
  [9/17] 2022-01-01 -> 2022-06-30 ... OK (4321 rows)
  [10/17] 2022-07-01 -> 2022-12-31 ... OK (4393 rows)
  [11/17] 2023-01-01 -> 2023-06-30 ... OK (4321 rows)
  [12/17] 2023-07-01 -> 2023-12-31 ... OK (4393 rows)
  [13/17] 2024-01-01 -> 2024-06-30 ... OK (4345 rows)
  [14/17] 2024-07-01 -> 2024-12-31 ... OK (4393 rows)
  [15/17] 2025-01-01 -> 2025-06-30 ... OK (4321 rows)
  [16/17] 2025-07-01 -> 2025-12-31 ... OK (4393 rows)
  [17/17] 2026-01-01 -> 2026-04-25 ... OK (2737 rows)
All 17 chunks downloaded.


## Step 3 - Concatenate and Clean

- Chunks are concatenated and deduplicated on the datetime column.
- `datetime (UTC)` is promoted to the index and renamed `datetime`.
- Metadata columns are dropped.

In [4]:
df = pd.concat(chunks, ignore_index=True)
df = df.drop_duplicates(subset='datetime (UTC)')

df['datetime (UTC)'] = pd.to_datetime(df['datetime (UTC)'])
df = df.set_index('datetime (UTC)').sort_index()
df.index.name = 'datetime'

df = df.drop(columns=[c for c in META_COLS if c in df.columns])

print('Shape      :', df.shape)
print('Date range :', df.index.min(), '->', df.index.max())
print('Columns    :', list(df.columns))
print()

missing = df.isnull().sum()
if missing.any():
    print('WARNING - Missing values:')
    print(missing[missing > 0])
else:
    print('OK - No missing values.')

display(df.head())

Shape      : (72497, 10)
Date range : 2018-01-01 00:00:00 -> 2026-04-25 00:00:00
Columns    : ['temperature (degC)', 'relative_humidity (0-1)', 'dewpoint_temperature (degC)', 'surface_solar_radiation (W/m^2)', 'surface_thermal_radiation (W/m^2)', 'total_precipitation (mm of water equivalent)', 'wind_speed (m/s)', 'wind_direction (deg)', 'skin_temperature (degC)', 'snowfall (mm of water equivalent)']

OK - No missing values.


,temperature (degC),relative_humidity (0-1),dewpoint_temperature (degC),surface_solar_radiation (W/m^2),surface_thermal_radiation (W/m^2),total_precipitation (mm of water equivalent),wind_speed (m/s),wind_direction (deg),skin_temperature (degC),snowfall (mm of water equivalent)
datetime,,,,,,,,,,
2018-01-01 00:00:00,7.15,0.92,5.91,0.0,320.89,0.03,2.49,202.77,6.29,0.0
2018-01-01 01:00:00,7.97,0.91,6.58,0.0,335.55,0.04,3.02,202.28,6.72,0.0
2018-01-01 02:00:00,7.89,0.88,6.05,0.0,336.98,0.03,3.74,198.00,7.42,0.0
2018-01-01 03:00:00,8.04,0.88,6.15,0.0,319.47,0.01,4.22,196.88,7.60,0.0
2018-01-01 04:00:00,8.12,0.88,6.20,0.0,334.21,0.03,4.08,198.42,7.38,0.0


## Step 4 - Save

Written with `index=True` so the datetime index is the first CSV column.
Notebook 01 reads it with `pd.read_csv(..., index_col=0, parse_dates=True)`.

In [5]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=True)
print(f'Saved {df.shape[0]} rows x {df.shape[1]} cols -> {OUTPUT_PATH}')

Saved 72497 rows x 10 cols -> ../data/raw/proxies/oikolab_weather.csv
